# LLM Benchmarking Example

This notebook provides an example to benchmark a LLM system with Ollama models. We'll cover:

1. Setting up the environment
2. Understanding the components
3. Running a simple benchmark
4. Analyzing results
5. Creating custom metrics

## 1. Environment Setup

First, make sure you have Ollama installed and running on your system. You can install it from [ollama.ai](https://ollama.ai).

Then, install the required Python packages:

```bash
pip install -r requirements.txt
```

You'll need to pull the models you want to use. For this example, we'll use llama3.2 and deepseek-r1 :

```bash
ollama pull llama3.2
ollama pull deepseek-r1:1.5b 
```

## 2. Understanding the Components

Let's import the main components of our system:

In [1]:
import sys
sys.path.append('.')  # Add parent directory to path

from models.ollama_model import OllamaModel
from metrics.relevance import RelevanceMetric
from core.run_benchmark import BenchmarkRunner
from metrics.metric_base import MetricResult
from metrics.evaluator import *

import asyncio


### 2.1 The Model Component

The `OllamaModel` class handles interactions with Ollama's API.

In [8]:
# Create a model instance
model = OllamaModel(model_name="llama3.2")

# Test the model with a simple prompt


async def test_model():
    response = await model.generate("What is the capital of France?")
    print(f"Response: {response.text}")
    print(f"Metadata: {response.metadata}")

await test_model()

Response: The capital of France is Paris.
Metadata: {'model': 'llama3.2', 'total_duration': 5408137499, 'load_duration': 3245790071, 'prompt_eval_duration': 1395821825, 'eval_duration': 764346140, 'eval_count': 8}


### 2.2 The Metrics Component

The `RelevanceMetric` class evaluates how relevant a response is to its prompt using another model.

In [10]:
# Create model and metric instances
model = OllamaModel(model_name="llama3.2")
evaluator_model = OllamaModel(model_name="deepseek-r1:1.5b")
evaluator = EvaluatorFactory.create_evaluator(evaluator_model)
metric = RelevanceMetric()

# Test the metric
async def test_metric():
    prompt = "What is the capital of France?"
    response = await model.generate(prompt)
    result: MetricResult = await metric.evaluate(prompt, response.text, evaluator=evaluator)
    print(f"Prompt: {prompt}")
    print(f"Response: {response.text}")
    print(f"Relevance Score: {result.score}")
    print(f"Rationale: {result.rationale}")
    print(f"Metadata: {result.metadata}")

await test_metric()

Prompt: What is the capital of France?
Response: The capital of France is Paris.
Relevance Score: 0.0
Rationale: Failed to parse evaluation response
Metadata: {'evaluation_method': 'ollama_llm', 'evaluator_model': 'deepseek-r1:1.5b', 'prompt_length': 30, 'response_length': 31}


## 3. Running a Benchmark

Now let's run a complete benchmark using the `BenchmarkRunner`:

In [2]:
# Models used to *evaluate* another model's output
jury_models = [
    # OllamaModel(model_name="gemma3:1b"),
    OllamaModel(model_name="llama3.2:latest")
]

# Create jury evaluator from the factory
evaluator = EvaluatorFactory.create_evaluator(jury_models)

# This model generates the response that will be judged
model_under_test = OllamaModel(model_name="deepseek-r1:1.5b")

metric = RelevanceMetric()

async def test_jury_evaluation():
    prompt = "Explain what quantum computing is in simple terms."
    response = await model_under_test.generate(prompt)

    result = await metric.evaluate(prompt, response.text, evaluator)

    print(f"Prompt: {prompt}")
    print(f"\nResponse from {model_under_test.model_name}:\n{response.text}")
    print(f"\nAverage Relevance Score: {result.score}")
    print(f"Combined Rationale:\n{result.rationale}")
    print(f"\nMetadata:\n{result.metadata}")

# Run the async test
await test_jury_evaluation()


Raw response from llama3.2:latest : 4 The response provides a clear explanation of quantum computing in simple terms, addressing key concepts such as qubits, superposition, entanglement, and their benefits over classical computers, while still acknowledging challenges in practical applications.
Prompt: Explain what quantum computing is in simple terms.

Response from deepseek-r1:1.5b:
<think>
Okay, so I'm trying to understand what quantum computing is, but it's a bit confusing because I know the term and hear some people talk about it a lot. Let me think this through step by step.

First, I remember hearing that quantum computers use something called qubits instead of regular bits. But wait, I don't fully get why they're different. Regular bits are like switches that can be either 0 or 1, right? So if there was a classical computer with just bits, it would handle things sequentially—like one operation at a time. But quantum computers use qubits which... Hmm, the user mentioned "nonclas

## 4. Analyzing Results

Let's load and analyze the benchmark results:

In [ ]:
import json
import pandas as pd
import plotly.express as px

# Load the results
with open(results['output_file'], 'r') as f:
    benchmark_results = json.load(f)

# Convert to DataFrame for easier analysis
df = pd.DataFrame(benchmark_results)

# Extract relevance scores
relevance_scores = [r['metrics']['relevance']['score'] for r in benchmark_results]

# Create a bar plot of relevance scores
fig = px.bar(
    x=range(len(relevance_scores)),
    y=relevance_scores,
    title='Relevance Scores by Prompt',
    labels={'x': 'Prompt Index', 'y': 'Relevance Score'}
)
fig.show()

# Display response times
response_times = [r['metadata']['total_duration'] for r in benchmark_results]
fig = px.bar(
    x=range(len(response_times)),
    y=response_times,
    title='Response Times by Prompt',
    labels={'x': 'Prompt Index', 'y': 'Response Time (ms)'}
)
fig.show()

## 5. Creating Custom Metrics

Let's create a simple custom metric that measures response length:

In [ ]:
from metrics.metric_base import BaseMetric, MetricResult

class LengthMetric(BaseMetric):
    """A simple metric that measures response length."""
    
    def __init__(self):
        super().__init__(
            name="length",
            description="Measures the length of the response in characters"
        )
    
    async def evaluate(self, prompt: str, response: str, evaluator) -> MetricResult:
        length = len(response)
        return MetricResult(
            score=length,
            rationale=f"Response length: {length} characters",
            metadata={"prompt_length": len(prompt)}
        )

# Test the custom metric
length_metric = LengthMetric()
result = await length_metric.evaluate(
    "What is the capital of France?",
    "Paris is the capital city of France."
)
print(f"Length score: {result.score}")
print(f"Rationale: {result.rationale}")

## 6. Next Steps

Here are some things you can try next:

1. Try different Ollama models (e.g., codellama, neural-chat)
2. Create more custom metrics
3. Implement more sophisticated evaluation methods
4. Create visualizations for other metrics
5. Add support for multi-turn conversations

The system is designed to be modular and extensible, so you can easily add new features and capabilities!